# SIH 26161 — Dam Break Inundation Pipeline (Colab)

One notebook to clone the repo and run the full pipeline:

1. **Setup** — clone repo, install the `dam_break` package.
2. **Scenario** — pick a dam (Machhu-II 1979 replay or South Lhonak/Teesta 2023) and set breach parameters.
3. **Run** — fetch DEM, compute the breach hydrograph, run the screening solver, export SHP/KML.
4. **Explore** — view maps, download outputs.
5. **Optional A** — Sentinel-1 near-real-time flood mapping in Google Earth Engine (Teesta 2023 event).
6. **Optional B** — DualSPHysics on the T4 GPU (near-field SPH).

> CPU runtime is enough for stages 1-4. Choose a GPU runtime
> (Runtime -> Change runtime type -> T4 GPU) only for stage 6.

## 1. Setup — clone the repo and install

In [ ]:
# Clone the repository (skipped if it already exists, e.g. after a reconnect)
![ -d SIH26161-dam-break-inundation ] || git clone https://github.com/akhileshkancharla/SIH26161-dam-break-inundation.git
%cd SIH26161-dam-break-inundation
!git pull --quiet 2>/dev/null || true

# Install dependencies + the package.
# NOT editable (-e): editable installs register their path via a .pth file
# that the already-running Colab kernel ignores (it is only read at
# interpreter startup), so `import dam_break` would fail without a restart.
# A regular install puts the package in site-packages and imports at once.
%pip install -q .

import os, sys
if os.path.isdir('src'):
    sys.path.insert(0, 'src')   # prefer the repo checkout over the installed copy
import dam_break
print('dam_break version:', dam_break.__version__)

## 2. Scenario — pick a dam and set parameters

The dam registry (`configs/datasets/dams_india.csv`) is a demo subset of
NRLD/GeoDAR. Attribute values marked `verified=False` are demo-grade —
check them against the National Register of Large Dams before quoting.

In [ ]:
import pandas as pd
from dam_break.ingestion.dams import load_registry

pd.set_option('display.width', 120)
registry = load_registry()
registry[['name', 'river', 'state', 'type', 'height_m', 'storage_mcm', 'verified']]

In [ ]:
# Choose the scenario: 'machhu2_1979' or 'teesta_south_lhonak_2023'
SCENARIO = 'machhu2_1979'

# Quick parameter overrides (see configs/scenarios/*.json for all knobs)
OVERRIDES = {
    'breach.case': 'expected',   # low | expected | high
    'breach.release_fraction': 0.6,
}

## 3. Run the pipeline end to end

In [ ]:
import json
from pathlib import Path
from dam_break.config import load_scenario
from dam_break.pipeline import run_pipeline

cfg = json.loads(Path(f'configs/scenarios/{SCENARIO}.json').read_text())

def apply_override(cfg, dotted, value):
    keys = dotted.split('.')
    node = cfg
    for k in keys[:-1]:
        node = node.setdefault(k, {})
    node[keys[-1]] = value

for k, v in OVERRIDES.items():
    apply_override(cfg, k, v)

summary = run_pipeline(load_scenario(cfg), out_root='outputs')

## 4. Explore the results

In [ ]:
# Breach summary
b = summary['breach']
print(f"Peak flow:        {b['peak_flow_m3s']:,.0f} m3/s")
print(f"Failure time:     {b['failure_time_min']:.0f} min")
print(f"Breach width:     {b['breach_width_m']:.0f} m")
print(f"Released volume:  {b['volume_m3']/1e6:.1f} Mm3")

d = summary['solvers']['screening']['diagnostics']
print(f"Inundated area:   {d['inundated_area_km2']:.1f} km2")
print(f"Path length:      {d['path_length_km']:.1f} km")
print(f"Front arrival at reach end: {d['arrival_last_min']:.0f} min")
for note in summary['solvers']['screening']['notes']:
    print('note:', note)

In [ ]:
# Preview maps
from IPython.display import Image, display
run_dir = Path(summary['run_dir'])
for png in sorted(run_dir.rglob('*.png')):
    print(png.relative_to(run_dir))
    display(Image(str(png), width=700))

In [ ]:
# Interactive map of the inundation polygons (depth classes)
import geopandas as gpd

gpkg = next((run_dir / 'screening').glob('*.gpkg'))
gdf = gpd.read_file(gpkg)
print(gdf.drop(columns='geometry').to_string(index=False))
gdf.explore(column='dclass', cmap='Blues', tooltip=['dclass', 'area_ha', 'maxdepth'])

In [ ]:
# Download the SHP (zip) and KML exports
from google.colab import files

for pattern in ('*.shp.zip', '*.kml'):
    for f in (run_dir / 'screening').glob(pattern):
        files.download(str(f))

## Full run: both solvers + exposure + loss + comparison (milestone 4)

One pipeline call with `solvers=["screening", "dualsphysics"]` and
`roughness.landcover="worldcover_gee"` produces: the corridor screening
raster, the near-field SPH raster, the automatic cross-solver comparison
(CSI/POD/FAR + depth bias + overlap map) and the land-cover exposure with
indicative depth-damage loss. Needs Earth Engine auth (WorldCover) and,
for the SPH half, the DualSPHysics package from Optional B's first cell.

In [ ]:
# 1) Earth Engine (WorldCover land cover + roughness)
%pip install -q earthengine-api
import ee
PROJECT = 'your-ee-project-id'   # <- your Google Cloud project registered with GEE
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('ee ok')

In [ ]:
# 2) Run screening + SPH with land-cover exposure in one go
#    (run Optional B cell 1 first if you want the SPH half to simulate)
import json
from pathlib import Path
from dam_break.config import load_scenario
from dam_break.pipeline import run_pipeline

cfg = json.loads(Path(f'configs/scenarios/{SCENARIO}.json').read_text())
cfg['roughness'] = {'landcover': 'worldcover_gee', 'default_n': 0.05}
cfg['solvers'] = ['screening', 'dualsphysics']
summary4 = run_pipeline(load_scenario(cfg), out_root='outputs')

In [ ]:
# 3) Comparison + loss tables
import json
import pandas as pd
from IPython.display import Image, display
from pathlib import Path

run4 = Path(summary4['run_dir'])
print('== cross-solver comparison ==')
print(json.dumps(summary4.get('comparison'), indent=1, default=str))
for png in sorted(run4.glob('comparison_*.png')):
    display(Image(str(png), width=700))

loss = (summary4.get('exposure') or {}).get('loss')
if loss:
    print('total indicative loss: Rs %.2f crore (%s)' % (
        loss['total_indicative_loss_crore_inr'], loss['caveat']))
    display(pd.DataFrame([{'class': k, **v} for k, v in loss['by_class'].items()]))
else:
    print('no land cover -> no loss table (check the ee auth cell)')

In [ ]:
# 4) Download the milestone-4 outputs
from google.colab import files
run4 = Path(summary4['run_dir'])
for pattern in ('comparison*.json', 'comparison*.png', 'screening/loss.csv',
                'screening/exposure.csv', 'screening/*.kml', 'dualsphysics/depth_max.tif'):
    for f in run4.rglob(pattern) if '/' in pattern else run4.glob(pattern):
        files.download(str(f))

## Optional A — Sentinel-1 flood mapping in GEE (near-real-time module)

Maps the actual 4 Oct 2023 South Lhonak GLOF extent around the lower Teesta
using the UN-SPIDER change-detection practice. Requires a Google account
authorized for Earth Engine (research/non-commercial use).

In [ ]:
# One-time authentication (follow the link), then initialize
import ee
from dam_break.satellite import init_ee, flood_mask, flood_area_km2

try:
    ee.Initialize(project='your-ee-project-id')   # <- set your Cloud project id
except Exception:
    ee.Authenticate()
    ee.Initialize(project='your-ee-project-id')

aoi = ee.Geometry.Rectangle([88.10, 27.25, 88.80, 27.95])  # Teesta below the lake
mask = flood_mask(
    aoi,
    before=('2023-09-15', '2023-09-30'),   # pre-event window
    after=('2023-10-05', '2023-10-12'),    # post-event window
)
print(f'flooded area: {flood_area_km2(mask, aoi):.1f} km2')

In [ ]:
# Visual check: S1 before/after and the flood mask
import geemap  # %pip install -q geemap if missing

before = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi) \
    .filterDate('2023-09-15', '2023-09-30').select('VV').median()
after = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi) \
    .filterDate('2023-10-05', '2023-10-12').select('VV').median()

Map = geemap.Map(center=[27.6, 88.45], zoom=9)
Map.addLayer(before, {'min': -25, 'max': 5}, 'S1 VV before')
Map.addLayer(after, {'min': -25, 'max': 5}, 'S1 VV after')
Map.addLayer(mask.selfMask(), {'palette': ['red']}, 'flood')
Map

## Optional B — DualSPHysics near-field simulation on the T4 GPU

The SPH solver covers the near-field reach only (dam + first ~1 km) — that
is by design (see the technical guide): SPH is the near-field/high-fidelity
solver, Delft3D FM the regional one.

Prerequisites:
- **GPU runtime**: Runtime -> Change runtime type -> T4 GPU.
- **The full DualSPHysics package zip in your Google Drive** (free download
  from dual.sphysics.org after a short registration). The public GitHub
  clone ships only GenCase + post tools, NOT the solver; the zip has the
  Linux solver binary.

Our adapter writes a real-terrain case (STL terrain in local coordinates,
sealed floor, terrain crust, rim-clamped reservoir), runs GenCase, executes
the solver (GPU when a driver is present, else CPU) and bins the fluid
particles into a georeferenced depth raster.

In [ ]:
# 1) Get the full DualSPHysics package (has the solver) onto this VM.
# Route A (no Drive mount needed): share the zip from drive.google.com as
#   "Anyone with the link", copy the FILE_ID from the share link, paste below.
# Route B: if you prefer mounting, comment the gdown block out and set ZIP
#   to your mounted path.
%pip install -q gdown
import gdown, os, zipfile
from pathlib import Path

FILE_ID = 'PASTE-YOUR-DRIVE-FILE-ID-HERE'   # share link: .../file/d/<FILE_ID>/view
ZIP = '/content/DualSPHysics_v5.4.3.zip'
root = Path('/content/DualSPHysics_v5.4')

if not root.exists():
    if 'PASTE-YOUR' not in FILE_ID and not Path(ZIP).exists():
        gdown.download(id=FILE_ID, output=ZIP)
    assert Path(ZIP).exists(), (
        'set FILE_ID to your shared Drive zip, or upload the zip manually '
        'and set ZIP to its path')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')

os.environ['DUALSPHYSICS_ROOT'] = str(root)
!chmod -R +x {root}/bin/linux

from dam_break.solvers.sph import find_tools
tools = find_tools()
print('gencase:', tools.gencase)
print('solver: ', tools.solver)
assert tools.solver is not None, 'solver binary not found - check the zip contents'
!nvidia-smi -L 2>/dev/null || echo 'WARNING: no GPU runtime - solver will use CPU'

In [ ]:
# 2) Build, validate and simulate the Machhu-II near-field case
#    dp trade-off: halving dp = 8x the particles. dp=3 keeps ~250k particles
#    (minutes on a T4); dp=2 gives ~900k (longer); start at 3.
import numpy as np
from pathlib import Path
from dam_break.config import load_scenario
from dam_break.ingestion.dem import fetch_dem
from dam_break.pipeline import _bbox_around, _dam_rowcol, _clip_to_corridor
from dam_break.breach import breach_parameters
from dam_break.solvers.sph import run_sph

sc = load_scenario(cfg)
dem = fetch_dem(sc.terrain, _bbox_around(sc.dam.lat, sc.dam.lon, 0.15))
rc = _dam_rowcol(dem, sc.dam.lat, sc.dam.lon)
dem, rc = _clip_to_corridor(dem, rc, 8.0, 4.0)

breach = breach_parameters(
    mode=sc.breach.mode, case=sc.breach.case, storage_m3=sc.dam.storage_m3,
    release_fraction=sc.breach.release_fraction, dam_height_m=sc.dam.height_m)

res = run_sph(dem, rc, breach, Path('outputs/sph_gpu'),
              dp=3.0, nearfield_length_m=800, timemax_s=60.0)
for k, v in res.diagnostics.items():
    print(f'{k}: {v}')
print()
print('\n'.join(res.notes))

In [ ]:
# 3) Inspect and download the SPH depth raster
from IPython.display import Image, display
from dam_break.postprocess import write_layer
from dam_break.postprocess.polygons import depth_polygons, export_vectors
from dam_break.visualize import plot_raster
from google.colab import files

case = Path(res.diagnostics['case_dir'])
layer = res.layers['depth_max']
write_layer(case / 'depth_max.tif', layer)
plot_raster(layer.array, layer.transform, case / 'depth_max.png',
            'SPH near-field max depth (GPU run)', 'm')
display(Image(str(case / 'depth_max.png'), width=700))

gdf = depth_polygons(layer.array, layer.transform, layer.crs)
export_vectors(gdf, case, 'sph_nearfield', ['kml', 'gpkg'], name='SPH nearfield')
for f in [case / 'depth_max.tif', case / 'sph_nearfield.kml', case / 'sph_nearfield.gpkg']:
    files.download(str(f))

## Optional C — Delft3D FM regional model (CPU, experimental)

The framework builds a complete D-Flow FM case (mesh + bed level +
reservoir initial level + native dambreak structure + boundary + obs
points). Executing it needs the Linux `dflowfm` binary; on Colab the
route is condacolab + the Deltares conda channel. This recipe is
experimental — verify channel/package names (tracked in
docs/verification-log.md) before relying on it.

In [ ]:
# 1) Build the case with our pipeline (no binary needed; self-contained)
from pathlib import Path
import rasterio.warp, numpy as np
from dam_break.config import load_scenario
from dam_break.ingestion.dem import fetch_dem
from dam_break.breach import breach_parameters, hydrograph_from_breach
from dam_break.pipeline import _bbox_around, _dam_rowcol, _clip_to_corridor
from dam_break.solvers.delft3d.adapter import run_delft3d

sc = load_scenario(cfg)
bbox = _bbox_around(sc.dam.lat, sc.dam.lon, sc.terrain.corridor_length_km * 1.6 / 111.0)
dem = fetch_dem(sc.terrain, bbox)
rc = _dam_rowcol(dem, sc.dam.lat, sc.dam.lon)
dem, rc = _clip_to_corridor(dem, rc, sc.terrain.corridor_length_km, sc.terrain.corridor_width_km)

breach = breach_parameters(
    mode=sc.breach.mode, case=sc.breach.case, storage_m3=sc.dam.storage_m3,
    release_fraction=sc.breach.release_fraction, dam_height_m=sc.dam.height_m)
t, q = hydrograph_from_breach(breach, dt_s=60.0, duration_h=sc.run.duration_h)

# 120 m mesh keeps Colab runtimes manageable; use 60 m for the final run
res = run_delft3d(dem, rc, breach, t, q, 0.05, Path('outputs/delft3d_manual'),
                  path_xy=None, duration_s=sc.run.duration_h * 3600,
                  mesh_resolution_m=120.0)
print(json.dumps(res.diagnostics, indent=1, default=str))
print('case ready at:', res.diagnostics['mdu'])

In [ ]:
# 2) (Experimental) install dflowfm via conda and run the case.
# NOTE: condacolab restarts the runtime once; re-run this cell afterwards.
%pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# After the condacolab restart, run this cell (same notebook):
!mamba install -y -c deltares delft3dfm
import os
os.environ['DFLOWFM_BIN'] = 'dflowfm'   # once installed on PATH

case_dir = 'outputs/delft3d_manual'     # from the build cell above
%cd {case_dir}
!dflowfm flowfm.mdu
# Map output: output/flowfm_map.nc -> post-process with the same
# dam_break.postprocess tools used for the screening solver.

## Next steps

- **Delft3D FM:** the case builder is done; execution needs `dflowfm`
  (Linux/WSL/Colab via conda, Optional C). Then compare arrival/peak/extent
  against the screening solver with `dam_break.postprocess.metrics`.
- **Full SPH runs (milestone 3):** run the generated case with the
  DualSPHysics binaries on the T4; keep `dp` >= 2 m and the domain <= 2 km.
- **Save outputs to Drive** so long runs survive session timeouts:
  `from google.colab import drive; drive.mount('/content/drive')` and pass
  `out_root='/content/drive/MyDrive/SIH26161'`.
- Everything run here is reproducible locally:
  `python scripts/run_pipeline.py configs/scenarios/machhu2_1979.json`.